# Code

In [1]:
import os
import cv2
import pickle
import dlib

def gather_landmark_data_dlib(directory):
    # Load dlib's face detector and landmark predictor
    detector = dlib.get_frontal_face_detector()
    predictor = dlib.shape_predictor('shape_predictor_68_face_landmarks.dat')  # Download this file
    
    landmark_data = {}
    skipped = 0
    processed = 0
    
    for class_name in os.listdir(directory):
        class_path = os.path.join(directory, class_name)
        if not os.path.isdir(class_path):
            continue
            
        for filename in os.listdir(class_path):
            img_path = os.path.join(class_path, filename)
            img = cv2.imread(img_path)
            
            if img is None:
                continue
            
            # Convert to grayscale
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            
            # Detect faces
            faces = detector(gray, 0)  # 0 means no upsampling
            
            if len(faces) > 0:
                face = faces[0]
                landmarks = predictor(gray, face)
                coords = []
                for i in range(68):
                    coords.append((landmarks.part(i).x, landmarks.part(i).y))
                landmark_data[img_path] = coords
                processed += 1
            else:
                skipped += 1
    
    print(f"Total: {processed} faces detected, {skipped} skipped")
    return landmark_data